In [1]:
from collections import Counter, defaultdict
import numbers
from pathlib import Path

import fitz
import numpy as np
import pandas as pd

pdf_base_dir = "path/to/PDFs Literaturdatenbank"
dev_set_coreschema_pdf_dir = f"{pdf_base_dir}/dev-set-100"
dev_set_organismtrend_pdf_dir = f"{pdf_base_dir}/dev-set-Wald-WVC"
test_set_coreschema_pdf_dir = f"{pdf_base_dir}/test"
test_set_organismtrend_pdf_dir = f"{pdf_base_dir}/test-set-AuO-WVC"
dev_set_coreschema_references = "../data/interim/faktencheck-db/faktenscheck_core_corrected.jsonl"
dev_set_coreschema_prediction_texts = "../data/processed/faktencheck/dev-set-100/predictions.jsonl"
dev_set_organismtrend_references = (
    "../data/external/organism_trends/Referenz_Wald_korrigiert_angepasst.csv"
)
dev_set_organismtrend_prediction_texts = (
    "../data/processed/faktencheck/dev-set-Wald-WVC/predictions.jsonl.gz"
)
test_set_coreschema_references = (
    "../data/interim/faktencheck-db/faktencheck-db-converted_2025-11-05.jsonl"
)
test_set_coreschema_prediction_texts = "../data/processed/faktencheck/test/predictions.jsonl.gz"
test_set_organismtrend_references = "../data/external/organism_trends/Weighted Vote Count Agrar- und Offenland Literatur - Sheet1.csv"
test_set_organismtrend_prediction_texts = (
    "../data/processed/faktencheck/test-set-AuO-WVC/predictions.jsonl.gz"
)

In [2]:
def count_column_values(col_name, value, val_counters, val_totals, val_sums):
    """Flattens lists/dicts and updates counters."""
    if value is None:
        return

    # dict → rekursiv flatten
    if isinstance(value, dict):
        for k, v in value.items():
            count_column_values(f"{col_name}.{k}", v, val_counters, val_totals, val_sums)
        return

    # list → jedes Element einzeln
    if isinstance(value, list):
        for item in value:
            count_column_values(col_name, item, val_counters, val_totals, val_sums)
        return

    # primitive value
    val_counters[col_name][value] += 1
    val_totals[col_name] += 1

    if isinstance(value, numbers.Number):
        val_sums[col_name] += value


def count_pages(df, col, base_dir):
    page_counts = []

    for file_path in df[col].dropna().unique():
        try:
            doc = fitz.open(Path(base_dir, file_path))
            num_pages = doc.page_count
            page_counts.append(num_pages)
            doc.close()

        except Exception as e:
            print(f"Error reading {file_path}: {e}")

    sum_pages = np.sum(page_counts) if page_counts else 0
    avg_pages = np.mean(page_counts) if page_counts else 0
    med_pages = np.median(page_counts) if page_counts else 0
    max_pages = max(page_counts) if page_counts else 0
    min_pages = min(page_counts) if page_counts else 0

    print(f"\n\nColumn: {col}")
    print(
        f"Total PDF files (in dataframe / with openable PDF): {len(df[col].dropna().unique())} / {len(page_counts)}"
    )
    print(f"Average pages per PDF: {avg_pages:.2f}")
    print(f"Median pages per PDF: {med_pages}")
    print(f"Max pages: {max_pages}")
    print(f"Min pages: {min_pages}")
    print(f"Total pages: {sum_pages}")


def count_words(df, col):
    word_counts = []

    for value in df[col].dropna():
        if not isinstance(value, str):
            continue

        words = value.split()  # whitespace split
        word_counts.append(len(words))

    # Summary stats
    sum_words = np.sum(word_counts) if word_counts else 0
    avg_words = np.mean(word_counts) if word_counts else 0
    med_words = np.median(word_counts) if word_counts else 0
    max_words = max(word_counts) if word_counts else 0
    min_words = min(word_counts) if word_counts else 0

    print(f"\n\nColumn: {col}")
    print(f"Average words per doc: {avg_words:.2f}")
    print(f"Median words per doc: {med_words}")
    print(f"Max words per doc: {max_words}")
    print(f"Min words per doc: {min_words}")
    print(f"Total words: {sum_words}")


def count_df_values(df, show_most_common=True):
    val_counters = defaultdict(Counter)
    val_totals = defaultdict(int)
    val_sums = defaultdict(float)

    for col in df.columns:
        for value in df[col]:
            count_column_values(col, value, val_counters, val_totals, val_sums)

    # Output
    for col in sorted(val_counters):
        print(f"\n=== {col} ===")
        print(f"Total values: {val_totals[col]}")

        if val_sums[col] != 0:
            print(f"Numeric sum: {val_sums[col]}")

        if show_most_common:
            for val, count in val_counters[col].most_common():
                print(f"  {val}: {count}")

In [3]:
# Dev set Core Schema reference stats
df = pd.read_json(dev_set_coreschema_references, lines=True)
print(df.head())

print("\n=== Unique gold entries (support) in the Dev-set CoreSchema reference data ===")
print(
    f"Unique docs with annotations: {df[["biodiversity_level", "ecosystem_type", "habitat", "taxa"]].notna().any(axis=1).sum()}"
)
count_df_values(
    df[["biodiversity_level", "ecosystem_type", "habitat", "taxa"]], show_most_common=False
)

  zotitem_ptr_id biodiversity_level  \
0       25ABQZIH               None   
1       25RIYD2C               None   
2       29Q84X7V               None   
3       2E9XWUUE               None   
4       2EUNPHDZ               None   

                                      ecosystem_type  \
0  [{'category': 'III Terrestrische und semiterre...   
1  [{'category': 'III Terrestrische und semiterre...   
2  [{'category': 'III Terrestrische und semiterre...   
3  [{'category': 'I Biotoptypengruppen der Meere ...   
4  [{'category': 'I Biotoptypengruppen der Meere ...   

                     habitat  \
0       Agrar- und Offenland   
1       Agrar- und Offenland   
2       Agrar- und Offenland   
3  Küsten und Küstengewässer   
4  Küsten und Küstengewässer   

                                                taxa  
0  [{'species_group': 'Insekten'}, {'species_grou...  
1                                               None  
2                                               None  
3  [{'species_g

In [4]:
# Dev set Core schema text stats
df = pd.read_json(dev_set_coreschema_prediction_texts, lines=True)
print(df.head())

print(f"\n\nTotal pdf texts in {dev_set_coreschema_prediction_texts}: {len(df['file_name'])}")

col = "text"
count_words(df, col)

col = "file_name"
count_pages(df, col, dev_set_coreschema_pdf_dir)

      file_name                                               text  \
0  25ABQZIH.pdf  # Informationspapier des Greifswald Moor Centr...   
1  25RIYD2C.pdf  27.08.25, 10:02 Grünes Band Deutschland: Chron...   
2  29Q84X7V.pdf  ## **WHAT WE CAN LEARN FROM THE GERMAN** **IMP...   
3  2E9XWUUE.pdf  # Neobiota der deutschen Nord- und Ostseeküste...   
4  2EUNPHDZ.pdf  [See discussions, stats, and author profiles f...   

   response_content  structured  structured_with_metadata  reasoning_content  \
0               NaN         NaN                       NaN                NaN   
1               NaN         NaN                       NaN                NaN   
2               NaN         NaN                       NaN                NaN   
3               NaN         NaN                       NaN                NaN   
4               NaN         NaN                       NaN                NaN   

   messages  messages_formatted errors errors_long  
0       NaN                 NaN     []       

In [5]:
# Dev set OrganismTrend Schema reference stats
df = pd.read_csv(dev_set_organismtrend_references)
df = df[df["Key"] != "#NV"]
print(df.head())

print(
    "\n=== Unique gold entries (support) in the OrganismTrend dev-set-Wald-WVC reference data ==="
)
print(f"Unique docs with annotations: {len(df['Key'].unique())}")

count_df_values(df[["Antwortvariable", "Lebensraum", "Trend", "Hauptgruppe_RoteListen"]])

     N            Key Bemerkung_Maria         Autor  Jahr  \
0    8       VJ4BEK2S         non_Sys  Bergerhausen  1981   
1   14       P2NC3SKD         non_Sys       Bücking  1989   
2   32  Ellwanger2014         non_Sys     Ellwanger  2014   
3   61       TYTG8U8R         non_Sys     Kostrzewa  1988   
4  111       K4ZL3L4P         non_Sys          Wolf  1989   

                                               Titel           Zeitschrift  \
0  Die Situation der Wiedereinbuergerung des Uhus...  Natur und Landschaft   
1  Bannwald Bechtaler Wald: Dauerbeobachtungen 19...  Natur und Landschaft   
2  Der nationale Bericht 2013 zu Lebensraumtypen ...  Natur und Landschaft   
3  Die Beeintraechtigung von Greifvogelhabitaten ...  Natur und Landschaft   
4  Probleme der Vegetationsentwicklung auf forstl...  Natur und Landschaft   

   DOI               Name_pdf  ID_Studie  ... Maßnahme_Schutzgebiet  \
0  NaN  Bergerhausen_1981.pdf        NaN  ...                   NaN   
1  NaN       Bücking_1

In [6]:
# Dev set OrganismTrend schema text stats
df = pd.read_json(dev_set_organismtrend_prediction_texts, lines=True)
print(df.head())

print(f"\n\nTotal pdf texts in {dev_set_organismtrend_prediction_texts}: {len(df['file_name'])}")

col = "text"
count_words(df, col)

col = "file_name"
count_pages(df, col, dev_set_organismtrend_pdf_dir)

      file_name                                               text  \
0  24ED3D2P.pdf  Journal of Vegetation Science 25 (2014) 113–12...   
1  26QI8J7B.pdf  Ecological Complexity 7 (2010) 260–272\n\n\n[C...   
2  2AWXR7KG.pdf                                                      
3  2CCR6MBI.pdf  [See discussions, stats, and author profiles f...   
4  2P53UVJA.pdf  ### **Invasion Ecology**\n\n\n# **Invasion Eco...   

   structured                      structured_with_metadata_list  \
0         NaN                                 [None, None, None]   
1         NaN                     [None, None, None, None, None]   
2         NaN                                                 []   
3         NaN                                 [None, None, None]   
4         NaN  [None, None, None, None, None, None, None, Non...   

                                         errors_list  \
0                                       [[], [], []]   
1                               [[], [], [], [], []]   
2 

In [7]:
# Test set Core Schema reference stats
df = pd.read_json(test_set_coreschema_references, lines=True)
# filter down to test set PDF ids
valid_test_set_docs = pd.read_json(test_set_coreschema_prediction_texts, lines=True)
df = df[df["zotitem_ptr_id"].isin(valid_test_set_docs["file_name"].str.removesuffix(".pdf"))]
print(df.head())

print("\n=== Unique gold entries (support) in the Test-set CoreSchema reference data ===")
print(
    f"Unique docs with annotations: {df[["biodiversity_level", "ecosystem_type", "habitat", "taxa"]].notna().any(axis=1).sum()}"
)
count_df_values(
    df[["biodiversity_level", "ecosystem_type", "habitat", "taxa", "biodiversity_variable"]],
    show_most_common=False,
)

                                        bibtex_author  \
1          Dörjes, J., Michaelis, H., &amp; Rhode, B.   
7   Polte, P., Gröhsler, T., Kotterba, P., von Nor...   
11  Bäcklin, B.-M., Persson, S., Faxneld, S., Rigé...   
20                                          Ebert, J.   
21  Beckmann, M., Gerstner, K., Akin-Fajiye, M., C...   

                                         bibtex_title bibtex_type bibtex_year  \
1   Long-term studies of macrozoobenthos in intert...     article        1986   
7   Reduced Reproductive Success of Western Baltic...     article        2021   
11  Temporal and Geographical Variation of Intesti...     article        2021   
20  Umsiedlungserfolg von Larven des Hirschkäfers:...     article        2011   
21  Conventional land‐use intensification reduces ...     article        2019   

                         biodiversity_level  \
1                           [Artenvielfalt]   
7                                      None   
11                           

In [8]:
# Test set Core schema text stats
df = pd.read_json(test_set_coreschema_prediction_texts, lines=True)
print(df.head())

print(f"\n\nTotal pdf texts in {test_set_coreschema_prediction_texts}: {len(df['file_name'])}")

col = "text"
count_words(df, col)

col = "file_name"
count_pages(df, col, test_set_coreschema_pdf_dir)

      file_name                                               text  \
0  256GWJWM.pdf  Hydrobiologia 142: 217 - 232 (1986)\n`O` Dr W....   
1  26F9N2EU.pdf  _Edited by:_\n\n_Pierluigi Carbonara,_\n\n_COI...   
2  27KRE5X4.pdf  # **_animals_**\n\n_Article_\n## **Temporal an...   
3  2AWXR7KG.pdf                                                      
4  2AYU97XZ.pdf  Received: 23 April 2018 Revised: 8 February 20...   

   structured   structured_with_metadata_list           errors_list  \
0         NaN              [None, None, None]          [[], [], []]   
1         NaN        [None, None, None, None]      [[], [], [], []]   
2         NaN              [None, None, None]          [[], [], []]   
3         NaN                              []                    []   
4         NaN  [None, None, None, None, None]  [[], [], [], [], []]   

           reasoning_content_list             character_start_list  \
0              [None, None, None]                [0, 19983, 39922]   
1        [No

In [9]:
# Test set OrganismTrend Schema reference stats
df = pd.read_csv(test_set_organismtrend_references)
df = df[df["Key"] != "#NV"]
print(df.head())

print(
    "\n=== Unique gold entries (support) in the OrganismTrend test-set-AuO-WVC reference data ==="
)
print(f"Unique docs with annotations: {len(df['Key'].unique())}")

count_df_values(df[["Antwortvariable", "Lebensraum", "Trend", "Hauptgruppe_RoteListen"]])

     N       Key Bemerkung_Maria                     Autor  Jahr  \
0   91  4EB2FXQT             NaN                   Schäfer  1993   
1  895  CWNR8562             NaN  Schmidt, Wolfgang et al.  1998   
2  894  CWNR8562             NaN  Schmidt, Wolfgang et al.  1998   
3  896  CWNR8562             NaN  Schmidt, Wolfgang et al.  1998   
4  874  NTJWRS7G             NaN    Evers, C, Zacharias, D  1999   

                                               Titel  \
0  Entwicklung und Ausbreitung von Amphibien-Popu...   
1  Straßenböschungen als Ersatzstandorte für Glat...   
2  Straßenböschungen als Ersatzstandorte für Glat...   
3  Straßenböschungen als Ersatzstandorte für Glat...   
4  Langzeitmonitoring primärer Binnensalzstellen ...   

                             Zeitschrift  DOI  ID_Studie        Screener  ...  \
0                   Natur und Landschaft  NaN        NaN  Maria Sporbert  ...   
1  Braunschweiger Geobotanische Arbeiten  NaN        NaN  Maria Sporbert  ...   
2  Braunsch

In [10]:
# Test set OrganismTrend schema text stats
df = pd.read_json(test_set_organismtrend_prediction_texts, lines=True)
print(df.head())

print(f"\n\nTotal pdf texts in {test_set_organismtrend_prediction_texts}: {len(df['file_name'])}")

col = "text"
count_words(df, col)

col = "file_name"
count_pages(df, col, test_set_organismtrend_pdf_dir)

      file_name                                               text  \
0  25RIYD2C.pdf  27.08.25, 10:02 Grünes Band Deutschland: Chron...   
1  26TU9DFJ.pdf  How to cite:\n\nKrausmann, Fridolin. “The Soci...   
2  27HEKAH2.pdf  Tuexenia 36: 395–412. Göttingen 2016.\ndoi: 10...   
3  28GB3CNK.pdf  Land Use Policy 54 (2016) 399–412\n\n\nContent...   
4  28NGZQDJ.pdf  Received: 13 February 2017 Revised: 30 May 201...   

   structured                     structured_with_metadata_list  \
0         NaN                                            [None]   
1         NaN                                            [None]   
2         NaN                          [None, None, None, None]   
3         NaN  [None, None, None, None, None, None, None, None]   
4         NaN                          [None, None, None, None]   

                        errors_list  \
0                              [[]]   
1                              [[]]   
2                  [[], [], [], []]   
3  [[], [], [], [], 